In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [3]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str
    evaluation: str

In [4]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']

    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline

    return state

In [5]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [6]:
def evaluate_blog(state: BlogState) -> BlogState:

    content = state['content']

    prompt = f'Evaluate the following blog content and provide a score out of 10 and a brief feedback \n {content}'

    evaluation = model.invoke(prompt).content

    state['evaluation'] = evaluation

    return state

In [7]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

workflow = graph.compile()

In [8]:
intial_state = {'title': 'Rise of AI in India'}

final_state = workflow.invoke(intial_state)

print(final_state)

{'title': 'Rise of AI in India', 'outline': 'Here\'s a detailed outline for a blog post on the "Rise of AI in India," designed to be comprehensive, engaging, and informative.\n\n---\n\n## Blog Title: The AI Tsunami: How India is Riding the Wave of Artificial Intelligence\n\n**Meta Description:** Explore the explosive growth of AI in India, from government initiatives and a booming startup ecosystem to its transformative impact across healthcare, finance, agriculture, and more. Discover the opportunities and challenges shaping India\'s AI future.\n\n---\n\n### I. Introduction: The AI Awakening in the Land of a Billion Dreams\n\n*   **A. Hook:** Start with a compelling statistic or a relatable scenario demonstrating AI\'s growing presence in daily Indian life (e.g., UPI payments, personalized e-commerce, smart assistants).\n*   **B. Defining AI (Briefly):** A quick, accessible explanation of what AI is, for a general audience.\n*   **C. Thesis Statement:** India is not just adopting AI; 

In [9]:
print(final_state['outline'])

Here's a detailed outline for a blog post on the "Rise of AI in India," designed to be comprehensive, engaging, and informative.

---

## Blog Title: The AI Tsunami: How India is Riding the Wave of Artificial Intelligence

**Meta Description:** Explore the explosive growth of AI in India, from government initiatives and a booming startup ecosystem to its transformative impact across healthcare, finance, agriculture, and more. Discover the opportunities and challenges shaping India's AI future.

---

### I. Introduction: The AI Awakening in the Land of a Billion Dreams

*   **A. Hook:** Start with a compelling statistic or a relatable scenario demonstrating AI's growing presence in daily Indian life (e.g., UPI payments, personalized e-commerce, smart assistants).
*   **B. Defining AI (Briefly):** A quick, accessible explanation of what AI is, for a general audience.
*   **C. Thesis Statement:** India is not just adopting AI; it's rapidly emerging as a global powerhouse in AI innovation,

In [10]:
print(final_state['content'])

## The AI Tsunami: How India is Riding the Wave of Artificial Intelligence

**Meta Description:** Explore the explosive growth of AI in India, from government initiatives and a booming startup ecosystem to its transformative impact across healthcare, finance, agriculture, and more. Discover the opportunities and challenges shaping India's AI future.

---

### I. Introduction: The AI Awakening in the Land of a Billion Dreams

From the seamless UPI payments that power our daily transactions to the personalized recommendations that shape our online shopping, and the intelligent voice assistants now speaking a multitude of Indian languages, Artificial Intelligence (AI) has quietly yet profoundly woven itself into the fabric of Indian life. It’s no longer a futuristic concept but a present-day reality, enhancing convenience, efficiency, and accessibility across the nation.

At its core, AI refers to the simulation of human intelligence in machines programmed to think like humans and mimic t

In [11]:
print(final_state['evaluation'])

## Evaluation:

**Score: 9.5/10**

### Brief Feedback:

This is an exceptionally well-researched, structured, and comprehensive blog post that effectively captures the multifaceted narrative of AI in India.

**Strengths:**

*   **Excellent Structure and Flow:** The logical progression from introduction to drivers, impact, ecosystem, challenges, and future outlook is superb. Headings and subheadings are clear, making the content easy to digest.
*   **Rich in Specific Examples:** The article doesn't just generalize; it provides concrete, relatable examples like UPI payments, the "Jio revolution," Aadhaar, NITI Aayog's "AI for All," and sector-specific applications (e.g., diabetic retinopathy in healthcare, pest detection in agriculture). This grounds the narrative and makes it highly credible.
*   **Balanced Perspective:** It adeptly highlights both the immense opportunities and the significant challenges (data privacy, skill gaps, ethical AI, infrastructure) facing India's AI journey, p